# Multiclass image classification — final submission

16x16 RGB images, upscaled to 32x32. Trained 3 CNNs from scratch (no pretrained weights),
then ensembled with pseudo-labeling and TTA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tensorflow', 'opencv-python-headless',
    'scikit-learn', 'pandas', 'matplotlib', 'seaborn'
], check=True)

In [ ]:
import os
import pickle
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

# saves memory on A100
tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
SEED = 42
IMG_SIZE = 32
PAD_SIZE = 4
NUM_CLASSES = 10
BATCH_SIZE = 128
VAL_SPLIT = 0.1
EPOCHS = 250
BASE_PATH = '/content/drive/MyDrive/kaggle-Competition'

# CIFAR-10 mean/std (dataset is similar enough)
CIFAR_MEAN = np.array([125.307, 122.950, 113.865], dtype=np.float32)
CIFAR_STD = np.array([62.993, 62.089, 66.705], dtype=np.float32)

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
with open(os.path.join(BASE_PATH, 'train_X_y.pkl'), 'rb') as f:
    X_raw, y_raw = pickle.load(f)

with open(os.path.join(BASE_PATH, 'test_X.pkl'), 'rb') as f:
    Xt_raw = pickle.load(f)

y_raw = y_raw.flatten().astype(np.int32)
print(X_raw.shape, Xt_raw.shape, y_raw.shape)

In [ ]:
# original images are 16x16, models work better at 32x32
def upscale(X, size):
    out = np.empty((len(X), size, size, 3), dtype=np.uint8)
    for i, img in enumerate(X):
        out[i] = cv2.resize(img, (size, size), interpolation=cv2.INTER_LANCZOS4)
    return out

X_f = upscale(X_raw, IMG_SIZE)
del X_raw
gc.collect()

Xt_f = upscale(Xt_raw, IMG_SIZE)
del Xt_raw
gc.collect()

y = y_raw.copy()
del y_raw
gc.collect()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_f, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED
)
print(len(X_tr), len(X_val))

MEAN_NORM = CIFAR_MEAN / 255.0
STD_NORM = CIFAR_STD / 255.0

In [ ]:
def sample_beta(alpha):
    # beta via two gammas (tf doesn't have beta directly)
    g1 = tf.random.gamma(shape=[], alpha=alpha)
    g2 = tf.random.gamma(shape=[], alpha=alpha)
    return g1 / (g1 + g2)


def cutmix(imgs, labs):
    imgs = tf.cast(imgs, tf.float32)
    bs = tf.shape(imgs)[0]
    lam = sample_beta(1.0)

    cut = tf.cast(tf.cast(IMG_SIZE, tf.float32) * tf.sqrt(1.0 - lam), tf.int32)
    cx = tf.random.uniform([], 0, IMG_SIZE, dtype=tf.int32)
    cy = tf.random.uniform([], 0, IMG_SIZE, dtype=tf.int32)

    x1 = tf.clip_by_value(cx - cut // 2, 0, IMG_SIZE)
    x2 = tf.clip_by_value(cx + cut // 2, 0, IMG_SIZE)
    y1 = tf.clip_by_value(cy - cut // 2, 0, IMG_SIZE)
    y2 = tf.clip_by_value(cy + cut // 2, 0, IMG_SIZE)

    mask_y = tf.logical_and(tf.range(IMG_SIZE) >= y1, tf.range(IMG_SIZE) < y2)
    mask_x = tf.logical_and(tf.range(IMG_SIZE) >= x1, tf.range(IMG_SIZE) < x2)
    mask_2d = tf.cast(
        tf.logical_and(mask_y[:, tf.newaxis], mask_x[tf.newaxis, :]),
        tf.float32
    )[tf.newaxis, :, :, tf.newaxis]

    idx = tf.random.shuffle(tf.range(bs))
    mixed = imgs * (1.0 - mask_2d) + tf.gather(imgs, idx) * mask_2d
    la = 1.0 - tf.cast((x2 - x1) * (y2 - y1), tf.float32) / float(IMG_SIZE ** 2)
    ml = la * labs + (1.0 - la) * tf.gather(labs, idx)
    return tf.cast(mixed, tf.float32), tf.cast(ml, tf.float32)


def mixup(imgs, labs):
    imgs = tf.cast(imgs, tf.float32)
    bs = tf.shape(imgs)[0]
    lam = sample_beta(0.4)
    idx = tf.random.shuffle(tf.range(bs))
    xi = lam * imgs + (1.0 - lam) * tf.cast(tf.gather(imgs, idx), tf.float32)
    li = lam * labs + (1.0 - lam) * tf.gather(labs, idx)
    return tf.cast(xi, tf.float32), tf.cast(li, tf.float32)

In [ ]:
def image_aug(x, l):
    x = tf.cast(x, tf.float32) / 255.0
    x = tf.image.resize_with_crop_or_pad(x, IMG_SIZE + PAD_SIZE * 2, IMG_SIZE + PAD_SIZE * 2)
    x = tf.image.random_crop(x, [IMG_SIZE, IMG_SIZE, 3])
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.1)
    x = tf.image.random_contrast(x, 0.9, 1.1)
    x = tf.clip_by_value(x, 0.0, 1.0)
    return tf.cast(x, tf.float32), tf.cast(l, tf.float32)


def batch_aug(x, l):
    # half the time cutmix, half mixup
    x, l = tf.cond(
        tf.random.uniform([]) < 0.5,
        lambda: cutmix(x, l),
        lambda: mixup(x, l),
    )
    return x, l


def make_train_ds(X, y):
    yoh = tf.one_hot(y, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((X, yoh))
    ds = ds.shuffle(8192, seed=SEED)
    ds = ds.map(image_aug, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.map(batch_aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def make_val_ds(X, y):
    yoh = tf.one_hot(y, NUM_CLASSES)

    def proc(x, l):
        return tf.cast(x, tf.float32) / 255.0, tf.cast(l, tf.float32)

    return (
        tf.data.Dataset.from_tensor_slices((X, yoh))
        .batch(512)
        .map(proc, num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )


def make_pred_ds(X):
    def proc(x):
        return tf.cast(x, tf.float32) / 255.0

    return (
        tf.data.Dataset.from_tensor_slices(X)
        .batch(512)
        .map(proc, num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )


tr_ds = make_train_ds(X_tr, y_tr)
vl_ds = make_val_ds(X_val, y_val)
tst_ds = make_pred_ds(Xt_f)

In [ ]:
def wide_basic_block(x, out_filters, stride=1, dropout_rate=0.0):
    in_filters = x.shape[-1]
    shortcut = x

    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.Activation('relu')(x)

    if stride != 1 or in_filters != out_filters:
        shortcut = layers.Conv2D(
            out_filters, 1, strides=stride, padding='same',
            use_bias=False, kernel_initializer='he_normal'
        )(x)

    x = layers.Conv2D(
        out_filters, 3, strides=stride, padding='same', use_bias=False,
        kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(5e-4)
    )(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.Activation('relu')(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(
        out_filters, 3, padding='same', use_bias=False,
        kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(5e-4)
    )(x)
    return layers.Add()([x, shortcut])


def wide_layer(x, out_filters, n_blocks, stride, dropout_rate=0.0):
    x = wide_basic_block(x, out_filters, stride, dropout_rate)
    for _ in range(n_blocks - 1):
        x = wide_basic_block(x, out_filters, 1, dropout_rate)
    return x


def build_wrn(depth=28, widen=10, dropout=0.3, name='WideResNet'):
    n = (depth - 4) // 6
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    mean = tf.constant(MEAN_NORM, dtype=tf.float32)
    std = tf.constant(STD_NORM, dtype=tf.float32)
    x = layers.Lambda(lambda t: (tf.cast(t, tf.float32) - mean) / std)(inp)

    x = layers.Conv2D(
        16, 3, padding='same', use_bias=False, kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(5e-4)
    )(x)
    x = wide_layer(x, 16 * widen, n, stride=1, dropout_rate=dropout)
    x = wide_layer(x, 32 * widen, n, stride=2, dropout_rate=dropout)
    x = wide_layer(x, 64 * widen, n, stride=2, dropout_rate=dropout)

    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)

    # float32 output for mixed precision
    out = layers.Dense(
        NUM_CLASSES, activation='softmax', dtype='float32',
        kernel_initializer='he_normal'
    )(x)
    return models.Model(inp, out, name=name)

In [ ]:
def res_block_v2(x, filters, stride=1):
    shortcut = x

    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(
        filters, 3, strides=stride, padding='same', use_bias=False,
        kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(
        filters, 3, padding='same', use_bias=False,
        kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, strides=stride, padding='same',
            use_bias=False, kernel_initializer='he_normal'
        )(shortcut)

    return layers.Add()([x, shortcut])


def build_resnet110(name='ResNet110'):
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    mean = tf.constant(MEAN_NORM, dtype=tf.float32)
    std = tf.constant(STD_NORM, dtype=tf.float32)
    x = layers.Lambda(lambda t: (tf.cast(t, tf.float32) - mean) / std)(inp)

    x = layers.Conv2D(
        16, 3, padding='same', use_bias=False, kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    for _ in range(18):
        x = res_block_v2(x, 16)

    x = res_block_v2(x, 32, stride=2)
    for _ in range(17):
        x = res_block_v2(x, 32)

    x = res_block_v2(x, 64, stride=2)
    for _ in range(17):
        x = res_block_v2(x, 64)

    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)

    out = layers.Dense(
        NUM_CLASSES, activation='softmax', dtype='float32',
        kernel_initializer='he_normal'
    )(x)
    return models.Model(inp, out, name=name)

In [ ]:
def train_model(model, name, tr_ds, vl_ds, epochs=EPOCHS):
    save_path = os.path.join(BASE_PATH, f'{name}.keras')
    steps_per_epoch = len(X_tr) // BATCH_SIZE

    # cosine decay was more stable than step schedule here
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.1,
        decay_steps=epochs * steps_per_epoch,
        alpha=0.001,
    )

    cbs = [
        keras.callbacks.ModelCheckpoint(
            save_path, monitor='val_accuracy', save_best_only=True, verbose=0
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=50,
            restore_best_weights=True, verbose=1
        ),
    ]

    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=lr_schedule, momentum=0.9, nesterov=True
        ),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'],
    )

    print(f'training {name}...')
    h = model.fit(tr_ds, validation_data=vl_ds, epochs=epochs, callbacks=cbs, verbose=1)
    best = max(h.history['val_accuracy'])
    print(f'{name} best val acc: {best:.4f}')
    return model, best

In [ ]:
# three different architectures for diversity
mA = build_wrn(depth=28, widen=10, name='ModelA_WRN28x10')
mA, accA = train_model(mA, 'ModelA_WRN28x10', tr_ds, vl_ds)

mB = build_wrn(depth=40, widen=4, name='ModelB_WRN40x4')
mB, accB = train_model(mB, 'ModelB_WRN40x4', tr_ds, vl_ds)

mC = build_resnet110(name='ModelC_ResNet110')
mC, accC = train_model(mC, 'ModelC_ResNet110', tr_ds, vl_ds)

print(f'A={accA:.4f}  B={accB:.4f}  C={accC:.4f}')

In [ ]:
def pseudo_round(model_list, X_base, y_base, X_test, threshold, rnd):
    print(f'pseudo-label round {rnd}, threshold={threshold}')
    ds_test = make_pred_ds(X_test)
    probs = np.mean([m.predict(ds_test, verbose=0) for m in model_list], axis=0)
    conf = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    mask = conf >= threshold
    print(f'adding {mask.sum()} samples')
    return (
        np.concatenate([X_base, X_test[mask]]),
        np.concatenate([y_base, preds[mask]]),
    )


def finetune_model(model, name, tr_ds, vl_ds, epochs=10, lr=1e-3):
    save_path = os.path.join(BASE_PATH, f'{name}.keras')
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy'],
    )
    cbs = [
        keras.callbacks.ModelCheckpoint(
            save_path, monitor='val_accuracy', save_best_only=True
        )
    ]
    h = model.fit(tr_ds, validation_data=vl_ds, epochs=epochs, callbacks=cbs, verbose=1)
    return model, max(h.history['val_accuracy'])


# round 1 — looser threshold
Xp1, yp1 = pseudo_round([mA, mB, mC], X_f, y, Xt_f, 0.95, 1)
tr_pl1 = make_train_ds(Xp1, yp1)
mA, _ = finetune_model(mA, 'ModelA_pl1', tr_pl1, vl_ds, epochs=10, lr=1e-3)
mB, _ = finetune_model(mB, 'ModelB_pl1', tr_pl1, vl_ds, epochs=10, lr=1e-3)
mC, _ = finetune_model(mC, 'ModelC_pl1', tr_pl1, vl_ds, epochs=10, lr=1e-3)
del Xp1, tr_pl1
gc.collect()

# round 2 — only high-confidence labels
Xp2, yp2 = pseudo_round([mA, mB, mC], X_f, y, Xt_f, 0.98, 2)
tr_pl2 = make_train_ds(Xp2, yp2)
mA, _ = finetune_model(mA, 'ModelA_pl2', tr_pl2, vl_ds, epochs=10, lr=5e-4)
mB, _ = finetune_model(mB, 'ModelB_pl2', tr_pl2, vl_ds, epochs=10, lr=5e-4)
mC, _ = finetune_model(mC, 'ModelC_pl2', tr_pl2, vl_ds, epochs=10, lr=5e-4)
del Xp2, tr_pl2
gc.collect()

In [ ]:
def tta_predict(model, X, n_passes=15):
    all_passes = []
    for i in range(n_passes):
        def aug_proc(x):
            x = tf.cast(x, tf.float32) / 255.0
            x = tf.image.resize_with_crop_or_pad(
                x, IMG_SIZE + PAD_SIZE * 2, IMG_SIZE + PAD_SIZE * 2
            )
            x = tf.image.random_crop(x, [IMG_SIZE, IMG_SIZE, 3])
            x = tf.image.random_flip_left_right(x)
            return x

        # map before batch, otherwise crop shape breaks
        ds = (
            tf.data.Dataset.from_tensor_slices(X)
            .map(aug_proc, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(512)
            .prefetch(tf.data.AUTOTUNE)
        )
        all_passes.append(model.predict(ds, verbose=0))

    return np.mean(all_passes, axis=0)


print('TTA...')
pA = tta_predict(mA, Xt_f)
pB = tta_predict(mB, Xt_f)
pC = tta_predict(mC, Xt_f)

In [ ]:
# grid search weights on val set
val_ds = make_pred_ds(X_val)
vA = mA.predict(val_ds, verbose=0)
vB = mB.predict(val_ds, verbose=0)
vC = mC.predict(val_ds, verbose=0)

best_acc = 0.0
best_w = (0.4, 0.3, 0.3)

for wA in np.arange(0.1, 0.81, 0.05):
    for wB in np.arange(0.1, 0.81, 0.05):
        wC = round(1.0 - wA - wB, 4)
        if wC < 0.05:
            continue
        acc = accuracy_score(y_val, (wA * vA + wB * vB + wC * vC).argmax(1))
        if acc > best_acc:
            best_acc = acc
            best_w = (float(wA), float(wB), float(wC))

wA, wB, wC = best_w
print(f'weights: A={wA:.2f} B={wB:.2f} C={wC:.2f}')
print(f'val acc (no TTA): {best_acc:.4f}')

y_pred_final = (wA * pA + wB * pB + wC * pC).argmax(axis=1)

v_ens = (
    wA * tta_predict(mA, X_val, 10)
    + wB * tta_predict(mB, X_val, 10)
    + wC * tta_predict(mC, X_val, 10)
)
final_val_acc = accuracy_score(y_val, v_ens.argmax(axis=1))
print(f'final val acc (TTA + ensemble): {final_val_acc:.4f}')

In [ ]:
sub = pd.DataFrame({'rowId': range(len(y_pred_final)), 'label': y_pred_final})
out_path = os.path.join(BASE_PATH, 'submission_FINAL.csv')
sub.to_csv(out_path, index=False)
print(out_path)
sub.head()